# _Python for Scientific Data Analysis_


#  AstroPy and AstroQuery

## Section 3: AstroQuery


## (_Note_ : Outages Affecting Astroquery) 

Currently -- 11/17/2026 -- CDS is experiencing a major outage.  It has wrecked havoc on my notes: fixing the notes has taken some time. The entire astroquery infrastructure is affected: I include information about mirror sites as a result.  It is possible that further outages will occur: please save a copy of this notebook so you can see the output when the queries are working.

A lot of what it means to "do" astronomy involves querying public databases and downloading data from these sources.   And there are a TON of them: HST, JWST, Gaia, 2MASS, Keck, etc.   Just wait until we have LSST.

Astronomy/astrophysics is a data-rich, highly accessible field (as science *should* be: not all fields are like this).   E.g. in principle, any rando anywhere in the world can download public data from many ground-based and basically all space-based observatories -- usually after some proprietary period of 6 to 18 months -- and make scientific discoveries.  

E.g. I know of a few cases where -- with some cursory background in coding, statistics and knowing how to navigate the Internet -- someone could have discovered a directly imaged planet from publicly accessible data!

So how do you access all of these data?   In many cases, you have to do it manually.   But for a lot of data --especially those contained in catalogues, fully reduced and analyzed -- you can use **astroquery**.  

Astroquery is a set of tools for querying astronomical web forms and databases.   It is an _affiliated package_ of AstroPy (more on that in Section 4).  For now, that means that you have to download and install astroquery separately from the core astropy package.  To do that, at Terminal you need to use either conda or pip ...

#pip

``pip install astroquery``

#conda 

``conda install -c conda-forge astroquery``

There are two other packages with complimentary functionality as Astroquery: pyvo is an Astropy affiliated package, and Simple-Cone-Search-Creator to generate a cone search service complying with the IVOA standard. They are more oriented to general virtual observatory discovery and queries, whereas Astroquery has web service specific interfaces.  

To use astroquery, we start with our same basic import functions we used for astropy:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

To these we add the astroquery import function

In [2]:
import astroquery

is the most basic import command.  But usually we want to import specific catalogues/servers from which astroquery will pull data.  E.g.

In [3]:
from astroquery.simbad import Simbad #Simbad astronomical database
from astroquery.vizier import Vizier #Vizier database

Let's start with these two catalog queries. These are some that are the most common ones you will want to deal with.  What are they?

* **Simbad** - SIMBAD (the Set of Identifications, Measurements and Bibliography for Astronomical Data) is an astronomical database of extrasolar objects beyond the Solar System.  [http://simbad.u-strasbg.fr/simbad/]()

* **Vizier** - The VizieR Catalogue Service is an astronomical catalog service.   [https://vizier.cds.unistra.fr/viz-bin/VizieR]()

The simplistic but helpful way of thinking about this is that Simbad will pull information for individual objects whereas Vizier details with different catalogues.  

### Major Types of Queries

There are multiple ways of querying these data sources for objects or data properties.   But two chief ones you will be interested in are ``query_object`` and ``query_region``.   The purpose of these are fairly straightforward to understand.   

To run, do 

``variable=Simbad.query_object([object name])`` 

``variable=Simbad.query_region([coordinates],r=[radius])``

etc.

For ``query_object``, the [object name] simply needs to be in single or double quotes (i.e. that it is a string variable).

You can then subscript the variable storing the query by keywords to extract parameters

### Timeouts and Mirror Sites 

Usually, you can run ``Simbad.query_object``, ``Vizier.query_object`` without having to think about fine-tuning the details of your function call.  However, the recent (current) CDS outage wrecks havoc on this thinking, requiring you to understand the use of mirror sites.

Nominally, both Simbad and Vizier point to a CDS URL: since CDS is down, nothing below will work unless you specify the site you want to point to.

In [4]:
#which site does function each point to?

print('The Simbad Server is ',Simbad.server)
print('The Vizier Server is ',Vizier.VIZIER_SERVER)

The Simbad Server is  simbad.cds.unistra.fr
The Vizier Server is  vizier.cds.unistra.fr


So this is an obvious problem.  Thus, we need to set the servers properly. Thankfully, there are a lot of mirror sites for both.  Let's try to use the CfA one.  Unfortunately, "use the CfA one" is actually complicated and astroquery version-specific.  I can explain later why this is the case.

But for now just take that the following will work for Simbad ...

#### Simbad Mirror Setting

In [5]:
#Simbad.server='simbad.harvard.edu' #for astroquery 0.4.11 this works
Simbad.SIMBAD_URL = 'https://simbad.harvard.edu/simbad/sim-script' #for astroquery 0.4.7 you need to do this

In [6]:
#verifying ...

starname='hr 8799'
test=Simbad.query_object(starname)
type(test) #if it is astropy.table.table.Table, if the query fails, you will get NoneType

test

MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,int32
HD 218396,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,1


#### Vizier Mirror Setting

In [7]:
Vizier.clear_cache() #when you have a lot of site problems/timeouts, make sure to set this.

#Vizier.VIZIER_SERVER='vizier.nao.ac.jp'
#Vizier.TIMEOUT=20 #s

#Vizier.VIZIER_SERVER = 'vizier.idia.ac.za'
#Vizier.TIMEOUT = 20

Vizier.VIZIER_SERVER = 'vizier.cfa.harvard.edu'
Vizier.TIMEOUT = 20

In [8]:
#note: it is possible this cell will not work.  In that case, choose a different Vizier mirror.

from astropy.coordinates import Angle
starname='hr 8799'


test=Vizier.query_object('hr 8799', catalog='II/246/out',radius=Angle(5,"arcsec")) #note: not specifying a catalog is NOT working right now: 11/11 3:44pm

#print(test[0]) # this will print out the table entry/ies for your query

J,eJ=(test[0]['Jmag'][0]),test[0]['e_Jmag'][0]
#type(J) #it's floating point

#here, the two [0] are a bit wonky: the first is to subscript the table list the second is to effectively convert a masked array to a float
#you cannot do a formatted print statement with a masked array
print('The J band photometry for {0:s} is {1:f} +/- {2:f}'.format(starname,test[0]['Jmag'][0],test[0]['e_Jmag'][0]))
test

The J band photometry for hr 8799 is 5.383000 +/- 0.027000


TableList with 1 tables:
	'0:II/246/out' with 15 column(s) and 1 row(s) 

### Querying Simbad



In [9]:
#convert from coordinates in simbad to a coordinate object
#c=coordinates.SkyCoord(Angle(var['RA']+' hours'),Angle(var['DEC'] + ' degrees'))

#A Simple Query

starname='hr 8799'  #HR 8799
result=Simbad.query_object(starname)
result
#result[0] #prints out the same thing here

MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,int32
HD 218396,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,1


after querying Simbad for this object, you can then return specific values listed:

In [10]:
print(result['RA'][0]) #result[0]['RA'] also works here.  it means you are selecting the 0th row of the 'RA' column
#'23 07 28.7157'

print(result['DEC'][0])
#'+21 08 03.310'

print(result['COO_BIBCODE'][0]) #bibliography code

#lists all of the column names
result.colnames

23 07 28.7157
+21 08 03.310
2020yCat.1350....0G


['MAIN_ID',
 'RA',
 'DEC',
 'RA_PREC',
 'DEC_PREC',
 'COO_ERR_MAJA',
 'COO_ERR_MINA',
 'COO_ERR_ANGLE',
 'COO_QUAL',
 'COO_WAVELENGTH',
 'COO_BIBCODE',
 'SCRIPT_NUMBER_ID']

Compare the information listed in ``result`` with a manual search for the same object on the Simbad webpage.   You will likely see that the webpage entry contains a lot more information missing in the table.  To be able to extract this information using astroquery, you need to add the variable in question to the possible fields.  

To figure out the fields you can use, execute ``Simbad.list_votable_fields()``, which lists the possible table fields.

In [11]:
print(Simbad.list_votable_fields())

--NOTES--

1. The parameter filtername must correspond to an existing filter. Filters include: B,V,R,I,J,K.  They are checked by SIMBAD but not astroquery.simbad

2. Fields beginning with rvz display the data as it is in the database. Fields beginning with rv force the display as a radial velocity. Fields beginning with z force the display as a redshift

3. For each measurement catalog, the VOTable contains all fields of the first measurement. When applicable, the first measurement is the mean one. 

Available VOTABLE fields:

bibcodelist(y1-y2)
biblio
cel
cl.g
coo(opt)
coo_bibcode
coo_err_angle
coo_err_maja
coo_err_mina
coo_qual
coo_wavelength
coordinates
dec(opt)
dec_prec
diameter
dim
dim_angle
dim_bibcode
dim_incl
dim_majaxis
dim_minaxis
dim_qual
dim_wavelength
dimensions
distance
distance_result
einstein
fe_h
flux(filtername)
flux_bibcode(filtername)
flux_error(filtername)
flux_name(filtername)
flux_qual(filtername)
flux_system(filtername)
flux_unit(filtername)
fluxdata(filtername)

You can then add fields using ``Simbad.add_votable_fields([field name])``.    Here are examples ...

In [12]:
Simbad.add_votable_fields('flux(V)','flux(J)','flux(H)','flux(K)') #adds J, H, and K band photometry from 2MASS
Simbad.add_votable_fields('sp') #adds the spectral type
Simbad.add_votable_fields('plx')


Then the revised query looks like the following ...

In [13]:
result=Simbad.query_object(starname)
print(result[0])
result

 MAIN_ID        RA           DEC      RA_PREC DEC_PREC COO_ERR_MAJA COO_ERR_MINA COO_ERR_ANGLE COO_QUAL COO_WAVELENGTH     COO_BIBCODE     FLUX_V FLUX_J FLUX_H FLUX_K  SP_TYPE   PLX_VALUE SCRIPT_NUMBER_ID
             "h:m:s"       "d:m:s"                         mas          mas           deg                                                   mag    mag    mag    mag                 mas                    
--------- ------------- ------------- ------- -------- ------------ ------------ ------------- -------- -------------- ------------------- ------ ------ ------ ------ ---------- --------- ----------------
HD 218396 23 07 28.7157 +21 08 03.310      14       14        0.037        0.040            90        A              O 2020yCat.1350....0G  5.953  5.383   5.28   5.24 F0+VkA5mA5    24.462                1


MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,FLUX_V,FLUX_J,FLUX_H,FLUX_K,SP_TYPE,PLX_VALUE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,mag,mag,mag,mag,,mas,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,float32,float32,float32,float32,object,float64,int32
HD 218396,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,5.953,5.383,5.28,5.24,F0+VkA5mA5,24.462,1


See that we've now added the four columns.    We can extract their values.  But first we need to know the names of teh columns to extract.  The mapping from the strings contained in "add votable fields" to the metadata for the table sometimes isn't one to one.  

Thankfully, you can see the names of the columns and then figure out which one corresponds to which intended field that you added.  See below ...

In [14]:
result[0].colnames

['MAIN_ID',
 'RA',
 'DEC',
 'RA_PREC',
 'DEC_PREC',
 'COO_ERR_MAJA',
 'COO_ERR_MINA',
 'COO_ERR_ANGLE',
 'COO_QUAL',
 'COO_WAVELENGTH',
 'COO_BIBCODE',
 'FLUX_V',
 'FLUX_J',
 'FLUX_H',
 'FLUX_K',
 'SP_TYPE',
 'PLX_VALUE',
 'SCRIPT_NUMBER_ID']

So, "Flux J" is the J band magnitude that we added as a votable field, etc etc.

Now, we can extract values

In [15]:
spt=result['SP_TYPE'][0]
#spt #'F0+VkA5mA5'

jmag=result['FLUX_J'][0]
hmag=result['FLUX_H'][0]
kmag=result['FLUX_K'][0]
plx=result['PLX_VALUE'][0]

print(jmag,hmag,kmag)
print(spt)
print(plx)

5.383 5.28 5.24
F0+VkA5mA5
24.462


The coordinate query returns strings for right ascension and declination.  With a bit of trickery you can turn this into a coordinate object in Astropy and save in units of degrees ...

In [16]:

import astropy.units as u
from astropy import coordinates 
from astropy.coordinates import SkyCoord
from astropy.coordinates import Angle

ra=result['RA'][0]
dec=result['DEC'][0]

c=coordinates.SkyCoord(Angle(ra +' hours'),Angle(dec + ' degrees'))
#c=coordinates.SkyCoord(Angle(ra +' hours'),Angle(dec + ' degrees'))
#c=coordinates.SkyCoord(Angle(ra),Angle(dec)) #this doesn't work

print(ra)
print(dec)
print('and now c ')
print(c)
print(c.ra)
print(c.dec)

23 07 28.7157
+21 08 03.310
and now c 
<SkyCoord (ICRS): (ra, dec) in deg
    (346.86964875, 21.13425278)>
346d52m10.7355s
21d08m03.31s


For a coordinate query, you are searching for objects around your target within some radius

In [17]:
result2=Simbad.query_region(c,radius=10*u.arcminute)

########to sort based on distance away, create a new SkyCoord object ...#####

coords=SkyCoord(result2['RA'],result2['DEC'],unit=(u.hourangle,u.deg))
angsep=c.separation(coords)*3600 #to arc-seconds 

###method 1
# Add the separation as a new column
#result2['separation'] = angsep
#result2.sort('separation')
#result2


##method 2
#result2.sort(angsep.argsort())
result2f=result2[angsep.argsort()]


In [18]:
result2f

MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,FLUX_V,FLUX_J,FLUX_H,FLUX_K,SP_TYPE,PLX_VALUE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,mag,mag,mag,mag,,mas,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,float32,float32,float32,float32,object,float64,int32
HD 218396c,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,--,--,--,--,,24.462,1
HD 218396b,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,--,--,--,--,,24.462,1
HD 218396e,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,--,--,--,--,,24.462,1
HD 218396d,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,--,--,--,--,,24.462,1
HD 218396f,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,--,--,--,--,,24.462,1
HD 218396,23 07 28.7157,+21 08 03.310,14,14,0.037,0.040,90,A,O,2020yCat.1350....0G,5.953,5.383,5.28,5.24,F0+VkA5mA5,24.462,1
TYC 1718-851-1,23 07 24.5842,+21 08 01.068,14,14,0.013,0.013,90,A,O,2020yCat.1350....0G,9.75,7.574,6.975,6.843,G0,2.100,1
SDSS J230725.90+210459.0,23 07 25.9082,+21 04 59.093,14,14,0.029,0.032,90,A,O,2020yCat.1350....0G,--,15.071,14.842,14.929,A0,0.144,1
